In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.match_results_football_data"

LEAGUES = ["bundesliga", "la_liga", "ligue_one", "premier_league", "serie_a"]


def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(
                    F.trim(F.col(f"`{column}`")) == "",
                    None
                ).otherwise(
                    F.trim(F.col(f"`{column}`"))
                )
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze tables
# ---------------------------------------------------------

dataframes = []

for league in LEAGUES:
    table_name = f"{BRONZE_SCHEMA}.football_data_{league}"

    if not spark.catalog.tableExists(table_name):
        print(f"WARNING: Table does not exist: {table_name}")
        continue

    df = spark.table(table_name)
    df = clean_strings(df)

    dataframes.append(df)


if not dataframes:
    raise ValueError("No Bronze tables were found.")


# ---------------------------------------------------------
# 2. Combine Bronze data
# ---------------------------------------------------------

df_combined = dataframes[0]

for df in dataframes[1:]:
    df_combined = df_combined.unionByName(
        df,
        allowMissingColumns=True
    )


# ---------------------------------------------------------
# 3. Select and standardize columns
# ---------------------------------------------------------

df_silver = df_combined.select(
    # Business columns - match information
    F.col("Div").cast("string").alias("division"),
    F.col("Date").cast("string").alias("match_date"),
    F.col("Time").cast("string").alias("kickoff_time"),
    F.col("HomeTeam").cast("string").alias("home_team"),
    F.col("AwayTeam").cast("string").alias("away_team"),

    F.col("FTHG").cast("int").alias("full_time_home_goals"),
    F.col("FTAG").cast("int").alias("full_time_away_goals"),
    F.col("FTR").cast("string").alias("full_time_result"),

    F.col("HTHG").cast("int").alias("half_time_home_goals"),
    F.col("HTAG").cast("int").alias("half_time_away_goals"),
    F.col("HTR").cast("string").alias("half_time_result"),

    # Business columns - match statistics
    #F.col("Attendance").cast("int").alias("attendance"),
    F.col("Referee").cast("string").alias("referee"),

    F.col("HS").cast("int").alias("home_shots"),
    F.col("AS").cast("int").alias("away_shots"),
    F.col("HST").cast("int").alias("home_shots_on_target"),
    F.col("AST").cast("int").alias("away_shots_on_target"),

    F.col("HC").cast("int").alias("home_corners"),
    F.col("AC").cast("int").alias("away_corners"),

    F.col("HF").cast("int").alias("home_fouls"),
    F.col("AF").cast("int").alias("away_fouls"),

    F.col("HY").cast("int").alias("home_yellow_cards"),
    F.col("AY").cast("int").alias("away_yellow_cards"),

    F.col("HR").cast("int").alias("home_red_cards"),
    F.col("AR").cast("int").alias("away_red_cards"),

    # Business columns - Bet365 pre-closing odds
    F.col("B365H").cast("double").alias("bet365_home_win_odds"),
    F.col("B365D").cast("double").alias("bet365_draw_odds"),
    F.col("B365A").cast("double").alias("bet365_away_win_odds"),

    # Context
    F.col("_season").cast("string").alias("_season"),
    F.col("_league").cast("string").alias("_league"),

    # Bronze metadata
    F.col("_ingested_at").cast("timestamp").alias("_bronze_ingested_at"),
    F.col("_source_file").cast("string").alias("_bronze_source_file"),
    F.col("_source_system").cast("string").alias("_source_system"),
)


# ---------------------------------------------------------
# 4. Remove duplicates
# ---------------------------------------------------------

duplicate_columns = [
    "_season",
    "_league",
    "home_team",
    "away_team"
]

duplicate_count = (
    df_silver
    .groupBy(duplicate_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate groups found: {duplicate_count}")

if duplicate_count > 0:
    df_silver = df_silver.dropDuplicates(duplicate_columns)
    print("Duplicates removed.")
else:
    print("No duplicates found. No changes made.")


# ---------------------------------------------------------
# 5. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 6. Basic validation
# ---------------------------------------------------------

null_team_names = df_silver.filter(
    F.col("home_team").isNull() | F.col("away_team").isNull()
).count()

if null_team_names > 0:
    print(f"WARNING: NULL team_name: {null_team_names}")

print(f"Silver rows: {df_silver.count()}")


# ---------------------------------------------------------
# 7. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")